# 01 · Data-quality check
Run weekly, not just at the end. Flags stale products (no snapshot in 48h — a broken selector looks like this), impossible values, duplicates, and scrape-log failures.

In [ ]:
import sys; sys.path.insert(0, "..")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path
from pricing.db import get_engine, load_snapshots, load_scrape_log
from pricing.clean import clean_snapshots, dim_products, gap_report
from pricing import analysis as an
from pricing.export import load_matches
raw = load_snapshots(get_engine())
clean, report = clean_snapshots(raw)
events = an.discount_events(clean)
print(f"{len(raw):,} raw rows -> {len(clean):,} clean rows, {clean.product_url.nunique()} products")

In [ ]:
pd.Series(report, name="rows")

In [ ]:
gaps = gap_report(clean)
print("Stale (no snapshot in 48h):", len(gaps["stale_products"]))
pd.DataFrame(gaps["stale_products"]).head(20)

In [ ]:
pd.DataFrame(gaps["historic_gaps"]).sort_values("gap_hours", ascending=False).head(20) if gaps["historic_gaps"] else "No gaps > 48h"

In [ ]:
# share of snapshots where no struck-through MRP was found: a sudden jump = broken MRP selector
raw["mrp_assumed"] = raw["parse_notes"].fillna("").str.contains("mrp_assumed_equal_price")
raw.assign(day=raw.scraped_at.dt.date).groupby(["platform", "day"]).mrp_assumed.mean().unstack(0).plot(figsize=(9, 3), title="Share of rows with MRP assumed");

In [ ]:
log = load_scrape_log(get_engine())
log.groupby(["platform", "status"]).size().unstack(fill_value=0) if len(log) else "scrape_log is empty"